# 08 - Stylometric-neural detector on HC3

DistilBERT augmented by the 14 stylometric features into its classifier
head (§3.3), run on HC3 before any of it reaches MAGE.

The point of this notebook is one number that can be compared with the two
development figures already recorded. On this same validation split the lexical
baseline reaches TPR@1%FPR of 0.791 and Binoculars reaches 0.896. A third
figure here says whether the detector and its training loop work before 40 GPU
runs are spent on MAGE.

Threshold calibration here uses the validation split itself, as notebook 02 and
the Binoculars HC3 check both did, so the three figures are comparable. On MAGE
the calibration and test sets are separate (§3.4).

## 1. Features for the training pairs

The detector trains on the length-matched pairs rather than the whole training
split, so those need their own features. Notebook 07 computed features for the
validation split only.

This saves them, so re-running the notebook skips the recompute.

In [1]:
import numpy as np
import pandas as pd

from logos.paths import DATA_DIR
from logos.project_data.hc3 import load_hc3, clean_hc3
from logos.project_data.preprocess import length_match, preprocess, train_val_split
from logos.features.stylometric import FEATURE_NAMES, features_frame

TRAIN_PATH = DATA_DIR / "hc3" / "features" / "stylometric_train.parquet"
VAL_PATH = DATA_DIR / "hc3" / "features" / "stylometric.parquet"

hc3 = preprocess(clean_hc3(load_hc3()))
train_df, val_df = train_val_split(hc3, val_fraction=0.15)

matched = length_match(train_df)
print(f"{len(matched):,} texts, {len(matched) // 2:,} pairs")
print(f"validation: {len(val_df):,} texts")

7,788 texts, 3,894 pairs
validation: 11,708 texts


In [2]:
if TRAIN_PATH.exists():
    train_features = pd.read_parquet(TRAIN_PATH)
    print(f"loaded {len(train_features):,} rows")
else:
    train_features = features_frame(matched["text"].tolist(), show_progress=True)

    # Kept beside the features so the next cell can check the rows line up
    train_features = train_features.assign(
        label=matched["label"].to_numpy(),
        domain=matched["domain"].to_numpy(),
    )

    TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
    train_features.to_parquet(TRAIN_PATH, index=False)
    print(f"saved {len(train_features):,} rows to {TRAIN_PATH}")

train_features[FEATURE_NAMES].describe().T.round(3)

  1,000 of 7,788
  2,000 of 7,788
  3,000 of 7,788
  4,000 of 7,788
  5,000 of 7,788
  6,000 of 7,788
  7,000 of 7,788
saved 7,788 rows to /Users/xwm517/Desktop/logos/data/hc3/features/stylometric_train.parquet


,count,mean,std,min,25%,50%,75%,max
sentence_length_mean,7788.0,23.577,10.362,6.167,18.000,21.750,26.556,231.000
sentence_length_sd,7788.0,9.797,6.529,0.000,6.266,8.437,11.574,100.500
mattr,7788.0,0.871,0.039,0.619,0.849,0.875,0.897,1.000
hapax_rate,7788.0,0.670,0.085,0.263,0.617,0.675,0.727,1.000
stopword_rate,7788.0,0.485,0.049,0.100,0.457,0.488,0.516,0.695
punctuation_rate,7788.0,0.026,0.010,0.004,0.019,0.024,0.030,0.192
noun_rate,7788.0,0.256,0.056,0.073,0.221,0.251,0.286,0.767
verb_rate,7788.0,0.184,0.033,0.041,0.163,0.184,0.205,0.333
past_tense_rate,7788.0,0.033,0.027,0.000,0.014,0.027,0.044,0.235
adverb_rate,7788.0,0.055,0.027,0.000,0.034,0.052,0.073,0.185


## 2. Check the rows line up

Both feature files are aligned to their split by position, not by a key. The
splits are deterministic, so that holds, but a silent reordering would train
the model on other texts' features. These asserts are cheap and the failure is
otherwise invisible.

In [3]:
val_features = pd.read_parquet(VAL_PATH)

assert len(train_features) == len(matched)
assert len(val_features) == len(val_df)
assert (train_features["label"].to_numpy() == matched["label"].to_numpy()).all()
assert (val_features["label"].to_numpy() == val_df["label"].to_numpy()).all()

print("train", train_features.shape, " val", val_features.shape)
print("all-zero rows:",
      (train_features[FEATURE_NAMES] == 0).all(axis=1).sum(), "train,",
      (val_features[FEATURE_NAMES] == 0).all(axis=1).sum(), "val")

train (7788, 16)  val (11708, 17)
all-zero rows: 0 train, 0 val


## 3. Standardise and build the datasets

Thirteen features are proportions between 0 and 1; Flesch reading ease runs
from about -100 to +120. Unstandardised it would dominate the other thirteen
at initialisation simply because of its units.

The standardiser is fitted on the training features only, so nothing from the
validation split reaches training. DistilBERT's own 768 outputs need no
scaling, since its final block ends in a LayerNorm.

In [4]:
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast

from logos.detectors.stylometric import StylometricDataset, StylometricDetector, score, train

MAX_LENGTH = 256
BATCH_SIZE = 32
EPOCHS = 3
SEED = 0

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

scaler = StandardScaler().fit(train_features[FEATURE_NAMES])
train_scaled = scaler.transform(train_features[FEATURE_NAMES])
val_scaled = scaler.transform(val_features[FEATURE_NAMES])

print("train mean", np.abs(train_scaled.mean(axis=0)).max().round(12))
print("val range", val_scaled.min().round(2), val_scaled.max().round(2))

W1103 09:09:31.793000 22695 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


device: mps
train mean 0.0
val range -22.28 39.96


In [5]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_data = StylometricDataset(
    texts=matched["text"],
    labels=matched["label"],
    features=train_scaled,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)
val_data = StylometricDataset(
    texts=val_df["text"],
    labels=val_df["label"],
    features=val_scaled,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)

print(f"{len(train_data):,} training texts in {len(train_loader)} batches")
print(f"{len(val_data):,} validation texts in {len(val_loader)} batches")

7,788 training texts in 244 batches
11,708 validation texts in 366 batches


## 4. Train

Hyperparameters follow Sun et al.: base learning rate 2e-5, 10% warm-up, three
epochs. The loss should fall each epoch.

Seeding fixes the head's initialisation and the shuffling. Note that this makes
the run repeatable in its folds and its starting weights, but not bit-for-bit
in its scores, because floating-point reductions on a GPU are not deterministic
by default.

In [6]:
torch.manual_seed(SEED)
np.random.seed(SEED)

model = StylometricDetector()
train(model, train_loader, epochs=EPOCHS, device=device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1: loss 0.2123
epoch 2: loss 0.0185
epoch 3: loss 0.0036


StylometricDetector(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=F

## 5. Score, calibrate and results

The threshold is the smallest score at which no more than 1% of human texts are
flagged (§3.4). TPR@1%FPR is then the share of ChatGPT texts caught at that
threshold.

In [7]:
from logos.evaluation.metrics import auroc, calibrate_threshold, fpr_at_threshold, tpr_at_threshold

scores = score(model, val_loader, device=device)
labels = val_df["label"].to_numpy()

tau = calibrate_threshold(scores, labels)
print(f"tau*        {tau:.6f}")
print(f"TPR@1%FPR   {tpr_at_threshold(scores, labels, tau):.4f}")
print(f"actual FPR  {fpr_at_threshold(scores, labels, tau):.4f}")
print(f"AUROC       {auroc(scores, labels):.4f}")
print()
print("for comparison on the same split")
print("  lexical baseline  0.791")
print("  Binoculars        0.896")

tau*        0.998506
TPR@1%FPR   0.9735
actual FPR  0.0099
AUROC       0.9991

for comparison on the same split
  lexical baseline  0.791
  Binoculars        0.896


In [8]:
# By domain, so a large domain cannot hide a failure in a small one
rows = []

for domain, part in val_df.groupby("domain"):
    at = part.index.to_numpy()
    rows.append({
        "domain": domain,
        "texts": len(part),
        "tpr": tpr_at_threshold(scores[at], labels[at], tau),
        "fpr": fpr_at_threshold(scores[at], labels[at], tau),
        "auroc": auroc(scores[at], labels[at]),
    })

pd.DataFrame(rows).set_index("domain").round(4)

,texts,tpr,fpr,auroc
domain,,,,
finance,1245,0.9896,0.0070,0.9998
medicine,372,1.0000,0.0000,1.0000
open_qa,663,0.8415,0.0461,0.9797
reddit_eli5,9193,0.9951,0.0087,0.9997
wiki_csai,235,0.9677,0.0631,0.9939


In [9]:
# Where the two classes sit, which says whether the threshold is cutting through
# a gap or through the middle of a cluster
for label, name in [(0, "human"), (1, "chatgpt")]:
    part = scores[labels == label]
    print(f"{name:8s} mean {part.mean():.4f}  p5 {np.percentile(part, 5):.4f}  "
          f"p95 {np.percentile(part, 95):.4f}")

human    mean 0.0627  p5 0.0002  p95 0.7114
chatgpt  mean 0.9985  p5 0.9993  p95 0.9998


## 6. Save

In [10]:
OUT_PATH = DATA_DIR / "hc3" / "scores" / "stylometric.parquet"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "label": labels,
    "domain": val_df["domain"].to_numpy(),
    "score": scores,
}).to_parquet(OUT_PATH, index=False)

print(f"saved {len(scores):,} scores to {OUT_PATH}")

saved 11,708 scores to /Users/xwm517/Desktop/logos/data/hc3/scores/stylometric.parquet
